# KPI Dictionary & Data Quality Profile
This notebook profiles the retail orders dataset against the supplied data dictionary. It checks completeness, uniqueness, validity, consistency, and freshness, then calculates core KPIs.

In [ ]:
import pandas as pd
from pathlib import Path
from datetime import date

DATA_FILE = Path('retail-orders-raw.csv')
df = pd.read_csv(DATA_FILE)
df.head()

## 1. Standardization

In [ ]:
df.columns = [c.strip().lower() for c in df.columns]
for c in ['customer_segment','city','category','payment_status']:
    if c in df.columns:
        df[c] = df[c].astype('string').str.strip()
df['customer_segment'] = df['customer_segment'].replace({'student':'Student','STUDENT':'Student','fresher':'Fresher','professional':'Professional'})
df['payment_status'] = df['payment_status'].str.title()
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['quantity_num'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce')
df

## 2. Data-quality checks

In [ ]:
today = pd.Timestamp.today().normalize()
required = ['order_id','order_date','customer_segment','city','category','quantity','unit_price','payment_status']
allowed_segments = {'Student','Fresher','Professional'}
allowed_categories = {'Learning Kit','Course Access','Mentor Session'}
allowed_status = {'Paid','Pending','Failed','Refunded'}

checks = {
    'Completeness - required fields': int(df[required].isna().sum().sum()),
    'Uniqueness - duplicate order_id rows': int(df['order_id'].duplicated(keep=False).sum()),
    'Validity - invalid date': int(df['order_date'].isna().sum() + ((df['order_date'] < pd.Timestamp('2025-01-01')) | (df['order_date'] > today)).fillna(False).sum()),
    'Validity - quantity <= 0 or non-numeric': int(df['quantity_num'].isna().sum() + (df['quantity_num'] <= 0).fillna(False).sum()),
    'Validity - negative unit price': int((df['unit_price'] < 0).fillna(False).sum()),
    'Validity - discount outside 0-100': int(((df['discount_pct'] < 0) | (df['discount_pct'] > 100)).fillna(False).sum()),
    'Consistency - segment outside allowed set': int((~df['customer_segment'].isin(allowed_segments)).fillna(True).sum()),
    'Consistency - category outside allowed set': int((~df['category'].isin(allowed_categories)).fillna(True).sum()),
    'Consistency - payment status outside allowed set': int((~df['payment_status'].isin(allowed_status)).fillna(True).sum()),
}
quality_report = pd.DataFrame({'check': checks.keys(), 'failures': checks.values()})
quality_report['status'] = quality_report['failures'].eq(0).map({True:'PASS',False:'FAIL'})
quality_report

## 3. Freshness check

In [ ]:
latest_date = df['order_date'].max()
freshness_days = (today - latest_date).days if pd.notna(latest_date) else None
print({'latest_order_date': latest_date, 'freshness_days': freshness_days, 'freshness_status': 'PASS' if freshness_days is not None and freshness_days <= 1 else 'FAIL'})

## 4. KPI calculations

In [ ]:
valid = df.drop_duplicates('order_id', keep='first').copy()
valid = valid[valid['quantity_num'].gt(0) & valid['unit_price'].ge(0)]
valid['discount_pct'] = valid['discount_pct'].fillna(0)
valid['gross_value'] = valid['quantity_num'] * valid['unit_price']
valid['net_value'] = valid['gross_value'] * (1 - valid['discount_pct']/100)
paid = valid[valid['payment_status'].eq('Paid')]

kpis = pd.Series({
    'Gross Order Value': valid['gross_value'].sum(),
    'Net Sales Value': paid['net_value'].sum(),
    'Paid Order Rate %': paid.shape[0] / valid.shape[0] * 100 if len(valid) else 0,
    'Average Order Value': paid['net_value'].mean() if len(paid) else 0,
    'Units Sold': paid['quantity_num'].sum(),
    'Average Discount %': valid['discount_pct'].mean(),
    'Refund Rate %': (valid['payment_status'].eq('Refunded').mean()*100),
    'Failure Rate %': (valid['payment_status'].eq('Failed').mean()*100),
    'Pending Order Rate %': (valid['payment_status'].eq('Pending').mean()*100),
    'Revenue per Unit': paid['net_value'].sum()/paid['quantity_num'].sum() if paid['quantity_num'].sum() else 0,
})
kpis.to_frame('value')

## 5. Conclusion
The source data intentionally contains quality issues such as a duplicate order ID, missing values, invalid quantity/discount values, a non-numeric quantity, and inconsistent capitalization. The KPI layer should only publish after critical quality failures are resolved or explicitly quarantined.